In [0]:
USE CATALOG workspace ;
--use DATABASE workspace ;

## 1.Departamentos 

In [0]:
--CTAS
CREATE OR REPLACE TABLE salud_plata.departamentos AS
select 
    DISTINCT
    concat(id_departamento,'-',fuente) AS clave_departamento
    ,fuente
    ,id_departamento
    ,nombre
    --indicadores de clalidad
    ,case 
        when id_departamento is null then true 
        else false 
    end as en_cuarentena
    ,current_time() as fecha_carga
from 
    (
    select 'hosa'as fuente , *
    from salud_bronce.departamentos_ha
    UNION ALL
    select 'hosb'as fuente,*
    from salud_bronce.departamentos_hb
    )

## 2.Medicos


In [0]:
CREATE OR REPLACE TABLE salud_plata.medicos AS
select 
    DISTINCT
    concat(split_part(id_medico,'-',2),'-',fuente) as clave_medico
    ,id_medico
    ,split_part(id_medico,'-',2) id_medico_origen
    ,nombre
    ,apellido
    ,concat(nombre,' ',apellido) nombre_completo
    ,especialidad
    ,id_departamento
    ,concat(id_departamento,'-',fuente) as clave_departamento
    ,try_cast(npi as BIGINT) as npi
    ,fuente
    ,case 
        when id_medico is null or id_departamento is null then true 
        else false 
    end as en_cuarentena
    ,current_time() as fecha_carga
    
from 
    (
    select 'hosa'as fuente , *
    from salud_bronce.medicos_ha
    UNION ALL
    select 'hosb'as fuente, *
    from salud_bronce.medicos_hb
    )


In [0]:
-- select column_name
-- from system.information_schema.columns
-- where  table_catalog ='workspace'
-- and table_schema='salud_bronce'
-- and table_name='atenciones_hb'
--workspace

##3.pacientes


In [0]:
CREATE OR REPLACE TABLE salud_plata.pacientes AS 
with pacientes as (
    select 'hosa'as fuente 
        ,id_paciente
        ,nombre
        ,segundo_nombre
        ,apellido
        ,nro_seguro_social
        ,telefono
        ,genero
        ,fecha_nacimiento
        ,direccion
        ,fecha_modificacion
    from salud_bronce.pacientes_ha
    UNION ALL
    select 'hosb'as fuente
        ,codigo as id_paciente
        ,p_nombre  as nombre
        ,s_nombre  as segundo_nombre
        ,p_apellido
        ,nro_seguro_social
        ,telefono
        ,genero
        ,fecha_nacimiento
        ,direccion
        ,fecha_actualizacion as fecha_modificacion
    from salud_bronce.pacientes_hb
)
select  DISTINCT
        concat(split_part(id_paciente,'-',2),'-',fuente) as clave_paciente
        ,id_paciente
        ,nombre
        ,segundo_nombre
        ,apellido
        ,nro_seguro_social
        ,telefono
        ,genero
        ,try_cast(fecha_nacimiento as date) as fecha_nacimiento 
        ,direccion
        ,try_cast(fecha_modificacion as date) as fecha_modificacion
        ,case 
            when id_paciente is null then true 
            else false 
        end as en_cuarentena
        ,current_time() as fecha_carga
from pacientes
where fuente ='hosb'


In [0]:
select getdate()

##4.Atenciones

In [0]:
CREATE OR REPLACE TABLE workspace.salud_plata.atenciones AS
SELECT DISTINCT
    CONCAT(id_atencion, '-', fuente)      AS clave_atencion,
    id_atencion,
    id_paciente,
    id_medico,
    id_departamento,
    CONCAT(id_paciente, '-', fuente)      AS clave_paciente,
    CONCAT(id_medico, '-', fuente)        AS clave_medico,
    CONCAT(id_departamento, '-', fuente)  AS clave_departamento,
    TRY_CAST(fecha_atencion AS DATE)      AS fecha_atencion,
    tipo_atencion,
    codigo_procedimiento,
    TRY_CAST(fecha_insercion AS DATE)     AS fecha_insercion,
    TRY_CAST(fecha_modificacion AS DATE)  AS fecha_modificacion,
    fuente,
    CASE WHEN id_atencion IS NULL OR id_paciente IS NULL OR fecha_atencion IS NULL
              OR LOWER(tipo_atencion) = 'null' THEN true ELSE false END AS en_cuarentena,
    current_timestamp() AS fecha_carga
FROM (
    SELECT *, 'hosa' AS fuente FROM workspace.salud_bronce.atenciones_ha
    UNION ALL
    SELECT *, 'hosb' AS fuente FROM workspace.salud_bronce.atenciones_hb
);

##5. Transacciones

In [0]:
CREATE OR REPLACE TABLE workspace.salud_plata.transacciones AS
SELECT DISTINCT
    CONCAT(id_transaccion, '-', fuente)   AS clave_transaccion,
    id_transaccion,
    id_atencion,
    id_paciente,
    id_medico,
    id_departamento,
    CONCAT(id_atencion, '-', fuente)      AS clave_atencion,
    CONCAT(id_paciente, '-', fuente)      AS clave_paciente,
    CONCAT(id_medico, '-', fuente)        AS clave_medico,
    CONCAT(id_departamento, '-', fuente)  AS clave_departamento,
    TRY_CAST(fecha_visita AS DATE)        AS fecha_visita,
    TRY_CAST(fecha_servicio AS DATE)      AS fecha_servicio,
    TRY_CAST(fecha_pago AS DATE)          AS fecha_pago,
    tipo_visita,
    ROUND(TRY_CAST(monto_facturado AS DOUBLE), 2) AS monto_facturado,
    tipo_monto,
    ROUND(TRY_CAST(monto_pagado AS DOUBLE), 2)    AS monto_pagado,
    id_reclamo,
    id_pagador,
    codigo_procedimiento,
    codigo_cie,
    linea_negocio,
    id_medicaid,
    id_medicare,
    TRY_CAST(fecha_insercion AS DATE)     AS fecha_insercion,
    TRY_CAST(fecha_modificacion AS DATE)  AS fecha_modificacion,
    fuente,
    CASE WHEN id_transaccion IS NULL OR id_atencion IS NULL OR id_paciente IS NULL
              OR fecha_visita IS NULL THEN true ELSE false END AS en_cuarentena,
    current_timestamp() AS fecha_carga
FROM (
    SELECT *, 'hosa' AS fuente FROM workspace.salud_bronce.transacciones_ha
    UNION ALL
    SELECT *, 'hosb' AS fuente FROM workspace.salud_bronce.transacciones_hb
);

##6.Reclamos

In [0]:
CREATE OR REPLACE TABLE workspace.salud_plata.reclamos AS
SELECT DISTINCT
    CONCAT(id_reclamo, '-', fuente)       AS clave_reclamo,
    id_reclamo,
    id_transaccion,
    id_paciente,
    id_atencion,
    id_medico,
    id_departamento,
    CONCAT(id_transaccion, '-', fuente)   AS clave_transaccion,
    CONCAT(id_paciente, '-', fuente)      AS clave_paciente,
    TRY_CAST(fecha_servicio AS DATE)      AS fecha_servicio,
    TRY_CAST(fecha_reclamo AS DATE)       AS fecha_reclamo,
    aseguradora,
    TRY_CAST(monto_reclamado AS DOUBLE)   AS monto_reclamado,
    TRY_CAST(monto_pagado AS DOUBLE)      AS monto_pagado,
    estado_reclamo,
    tipo_pagador,
    TRY_CAST(deducible AS DOUBLE)         AS deducible,
    TRY_CAST(coaseguro AS DOUBLE)         AS coaseguro,
    TRY_CAST(copago AS DOUBLE)            AS copago,
    TRY_CAST(fecha_insercion AS DATE)     AS fecha_insercion,
    TRY_CAST(fecha_modificacion AS DATE)  AS fecha_modificacion,
    fuente,
    CASE WHEN id_reclamo IS NULL OR id_paciente IS NULL OR id_transaccion IS NULL
              OR LOWER(estado_reclamo) = 'null' THEN true ELSE false END AS en_cuarentena,
    current_timestamp() AS fecha_carga
FROM (
    SELECT *, 'hosa' AS fuente FROM workspace.salud_bronce.reclamos_ha
    UNION ALL
    SELECT *, 'hosb' AS fuente FROM workspace.salud_bronce.reclamos_hb
);

##7.CPT

In [0]:
CREATE OR REPLACE TABLE workspace.salud_plata.codigos_cpt AS
SELECT DISTINCT
    CONCAT(codigo_cpt, '-', 'hosa')  AS clave_cpt,
    categoria_procedimiento,
    codigo_cpt,
    descripcion_procedimiento,
    CASE WHEN LOWER(estado_codigo) = 'sin cambios' THEN 'Sin Cambios' ELSE estado_codigo END AS estado_codigo,
    'hosa' AS fuente,
    CASE WHEN codigo_cpt IS NULL OR LOWER(estado_codigo) = 'null' THEN true ELSE false END AS en_cuarentena,
    current_timestamp() AS fecha_carga
FROM workspace.salud_bronce.codigos_cpt;

In [0]:
SELECT 'departamentos' AS tabla, COUNT(*) AS filas, SUM(CAST(en_cuarentena AS INT)) AS en_cuarentena FROM workspace.salud_plata.departamentos
UNION ALL SELECT 'medicos',       COUNT(*), SUM(CAST(en_cuarentena AS INT)) FROM workspace.salud_plata.medicos
UNION ALL SELECT 'pacientes',     COUNT(*), SUM(CAST(en_cuarentena AS INT)) FROM workspace.salud_plata.pacientes
UNION ALL SELECT 'atenciones',    COUNT(*), SUM(CAST(en_cuarentena AS INT)) FROM workspace.salud_plata.atenciones
UNION ALL SELECT 'transacciones', COUNT(*), SUM(CAST(en_cuarentena AS INT)) FROM workspace.salud_plata.transacciones
UNION ALL SELECT 'reclamos',      COUNT(*), SUM(CAST(en_cuarentena AS INT)) FROM workspace.salud_plata.reclamos
UNION ALL SELECT 'codigos_cpt',   COUNT(*), SUM(CAST(en_cuarentena AS INT)) FROM salud_plata.codigos_cpt;